# 00 - EDA: Chest_x_ray

Este notebook sigue la misma estructura estándar que se usará para el EDA de cada fuente candidata. Las secciones son iguales en todos los notebooks de `notebooks_eda/` — de hecho las partes verdaderamente comunes (verificación de integridad, heurísticas de calidad de imagen, hashing MD5, funciones de graficado) viven en `eda_utils.py` y se importan; lo que cambia de un dataset a otro es el código de las secciones 2 (mapeo de etiquetas) y 7 (metadata), según si la fuente viene organizada en carpetas por clase, con un CSV de etiquetas, o con archivos de segmentación.

**Esta fuente en particular**: ya viene organizada en `train/val/test` con una carpeta por clase (`NORMAL`, `PNEUMONIA`, `TURBERCULOSIS`, `COVID19`) — es el dataset base ya usado en la versión IMJETA del artículo, por lo que estos números ya son conocidos, pero se vuelven a calcular aquí para dejar el mismo pipeline reproducible sobre las 6+ fuentes.


## 0. Configuración

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")  # para poder importar eda_utils.py (vive en este mismo directorio)
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "Chest_x_ray"
DATASET_DIR = Path("../Datasets/Chest_x_ray")
SPLITS = ["train", "val", "test"]

# Carpeta de origen -> clase objetivo del proyecto (4 clases actuales)
CLASS_MAP = {
    "NORMAL": "Normal",
    "PNEUMONIA": "Pneumonia",
    "TURBERCULOSIS": "Tuberculosis",  # nombre de carpeta viene así de origen (typo del dataset)
    "COVID19": "COVID-19",
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Carpeta compartida donde cada notebook de EDA guarda sus hashes MD5,
# para poder comparar solapamiento entre TODAS las fuentes más adelante
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico
Cuántos archivos hay, en qué carpeta (split/clase de origen), qué extensión y peso.

In [ ]:
records = []
for split in SPLITS:
    split_dir = DATASET_DIR / split
    if not split_dir.exists():
        print(f"Aviso: no existe {split_dir}")
        continue
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for f in eu.list_images(class_dir):
            records.append({
                "split": split,
                "source_class": class_dir.name,
                "path": str(f),
                "ext": f.suffix.lower(),
                "size_bytes": f.stat().st_size,
            })

df = pd.DataFrame(records)
print(f"Total de imágenes encontradas: {len(df)}")
df["ext"].value_counts()

In [ ]:
df.groupby("source_class")["size_bytes"].agg(["count", "mean", "min", "max"])

## 2. Esquema de etiquetas y mapeo a clases objetivo
Aquí no hay CSV: la propia carpeta ES la etiqueta. Se mapea directamente con `CLASS_MAP`.

In [ ]:
df["target_class"] = df["source_class"].map(CLASS_MAP)

unmapped = df[df["target_class"].isna()]
if len(unmapped):
    print("Carpetas sin mapear (revisar CLASS_MAP):", sorted(unmapped["source_class"].unique()))
else:
    print("Todas las carpetas de origen quedaron mapeadas correctamente.")

df = df.dropna(subset=["target_class"]).copy()

## 3. Conteo real por clase objetivo (por split)

In [ ]:
summary = df.groupby(["split", "target_class"]).size().unstack(fill_value=0)
summary = summary.reindex(columns=sorted(summary.columns))
summary.loc["Total"] = summary.sum()
summary["Total"] = summary.sum(axis=1)
summary.to_csv(OUTPUT_DIR / "conteo_por_clase.csv")
summary

In [ ]:
totales = df["target_class"].value_counts().sort_index()
ax = totales.plot(kind="bar", figsize=(6, 4), color="#4C72B0")
ax.set_title(f"{DATASET_NAME}: total de imágenes por clase (train+val+test)")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase.png", dpi=120)
plt.show()

## 4. Calidad e integridad
Verificar que cada imagen abra correctamente (descartar corruptas) y revisar la distribución de dimensiones y modo de color (escala de grises vs RGB).

In [ ]:
probe_results = [eu.probe_image(p) for p in tqdm(df["path"], desc="Verificando imágenes")]
df["width"], df["height"], df["mode"], df["error"] = zip(*probe_results)

corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no abribles: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Modo de color:")
display(df["mode"].value_counts())
print()
print("Dimensiones (ancho x alto):")
df[["width", "height"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["width"].hist(bins=40, ax=axes[0])
axes[0].set_title("Distribución de ancho (px)")
df["height"].hist(bins=40, ax=axes[1])
axes[1].set_title("Distribución de alto (px)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "distribucion_dimensiones.png", dpi=120)
plt.show()

### 4.1 Candidatas a excluir por calidad de imagen (revisión manual, no se borra nada aquí)


**Este notebook no decide nada por sí solo.** Se calculan dos heurísticas objetivas (`eda_utils.flag_quality_issues`), se guarda una lista de candidatas (`candidatas_baja_calidad.csv`) y una cuadrícula visual para que la exclusión final se revise caso por caso, con la razón documentada. Esa es la diferencia entre "filtrar por conveniencia" (poco ético) y "aplicar un criterio de calidad objetivo, consistente entre todas las clases, y documentado" (práctica estándar)

- **Heurística de bordes/recorte**: recorta filas y columnas con muy poca variación (uniformes, señal de margen en blanco o negro) y calcula qué fracción del área total es contenido real.
- **Heurística de brillo/reflejo**: mide qué fracción del *centro* de la imagen es casi blanco puro — un proxy de reflejo de luz, típico de fotografiar una placa física en vez de exportarla digital.

Ambas son aproximaciones simples con falsos positivos/negativos — por eso el resultado es una lista para revisar, no un filtro automático.

In [ ]:
calidad = eu.flag_quality_issues(df[df["error"].isna()], desc=DATASET_NAME)

sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
print()
print("Por clase:")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "split", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)
sospechosas[["path", "target_class", "split", "content_ratio", "glare_fraction", "motivo"]].sort_values(
    ["target_class", "content_ratio"]
)

In [ ]:
eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png")

**Sobre la idea de "quizás estas imágenes reflejan condiciones reales de despliegue"**: es un punto válido, y justo por eso no se borra nada automáticamente aquí. Es una decisión de diseño de investigación que conviene tomar conscientemente (no por accidente) y documentar en Materials and Methods, por ejemplo: (a) excluirlas del train/val por completo, o (b) excluirlas del entrenamiento pero conservar una pequeña submuestra en el test, etiquetada explícitamente como "subconjunto de robustez ante condiciones de adquisición no ideales" — relevante para el contexto de Apurímac si en la práctica alguien llega a subir una foto de una radiografía física en vez de una exportación digital. Queda pendiente de decidir una vez que se vea esta misma cuadrícula para las demás fuentes.

## 5. Muestra visual por clase
Grid de ejemplos aleatorios para inspección rápida (texto/marcas visibles, orientación, recortes raros).

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png")

## 6. Riesgo de solapamiento / duplicados
Hash MD5 exacto por imagen: detecta duplicados dentro del propio dataset (incluyendo fuga entre train/val/test) y se guarda para cruzarlo después contra las demás fuentes en un notebook de consolidación (mismo mecanismo que ya usa `01_dataset_preprocessing.ipynb`).

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
df["phash"] = [eu.phash_file(p) for p in tqdm(df["path"], desc="Hash perceptual")]

n_dupes = df["md5"].duplicated().sum()
print(f"Duplicados exactos dentro de este dataset: {n_dupes}")

splits_por_hash = df.groupby("md5")["split"].nunique()
leak_hashes = splits_por_hash[splits_por_hash > 1].index
print(f"Hashes que aparecen en más de un split (posible fuga train/val/test): {len(leak_hashes)}")

### 6.1 ¿Cuáles imágenes se duplican y con cuáles?
Tabla completa de los grupos duplicados, y además la comparación visual de los que cruzan splits (los más importantes, porque implican fuga de datos entre train/val/test).

In [ ]:
grupos_duplicados = (
    df[df.duplicated("md5", keep=False)]
    .sort_values("md5")[["md5", "path", "target_class", "split"]]
)
print(f"Grupos de imágenes duplicadas: {grupos_duplicados['md5'].nunique()}")
grupos_duplicados

In [ ]:
print(f"Mostrando los {len(leak_hashes)} grupos que cruzan splits (fuga real):")
for h in leak_hashes:
    print(f"Hash {h}:")
    eu.show_duplicate_group(df[df["md5"] == h], label_fn=lambda r: f"{r['split']}/{r['target_class']}\n{Path(r['path']).name}")

### 6.2 ¿Para qué sirve `Chest_x_ray_md5.csv`?

Es una "huella digital" por imagen (ruta + hash MD5 + clase + split), guardada aparte de este notebook. Sirve para un paso posterior de **consolidación entre TODAS las fuentes**: cuando ya exista el mismo archivo para TBX11K, RSNA, SilicoData, etc., un notebook de consolidación puede juntar todos estos CSV pequeños y buscar hashes repetidos **entre datasets distintos** — por ejemplo, si una imagen de Shenzhen/Montgomery ya está también dentro de TBX11K-simplified, un riesgo real que ya se había anticipado. Sin este archivo habría que volver a calcular el hash de las ~7,000 imágenes cada vez que se quiera comparar contra una fuente nueva — con él, ese cálculo se hace una sola vez por fuente y se reutiliza. `to_csv` sobrescribe el archivo cada vez que se corre esta celda, así que no se duplica ni acumula nada al re-ejecutar el cuaderno.

In [ ]:
# Se guarda para el cruce posterior entre TODAS las fuentes (incluyendo TBX11K, COVID-QU-Ex, etc.)
# md5 detecta copias byte-idénticas; phash (perceptual) detecta la misma radiografía aunque haya
# sido reconvertida/reescalada a otro formato en otra fuente (caso plausible entre distintos
# datasets de neumonía/COVID que reutilizan las mismas imágenes de origen).
df[["path", "md5", "phash", "target_class", "split"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_md5.csv", index=False
)

## 7. Metadata demográfica / poblacional

**No disponible para esta fuente**: son solo carpetas de imágenes ya organizadas por clase, sin CSV ni metadata de paciente (edad, sexo, vista de proyección, equipo).

Lo que sí se sabe por la documentación de las sub-fuentes originales (no verificable desde los archivos en sí, solo por literatura ya citada en el artículo V2):
- **Normal / Pneumonia**: población **pediátrica** (Kermany et al. 2018, niños de 1 a 5 años, Guangzhou, China).
- **Tuberculosis**: población adulta (Rahman et al. 2020).
- **COVID-19**: población adulta (Rahman et al. 2022).

Esta heterogeneidad de población/origen por clase es precisamente el riesgo de *domain shift* ya documentado en la discusión del artículo V2 (IMJETA) — se vuelve a dejar constancia aquí porque es información que no aparece en los archivos, solo en los papers de origen.

## 8. Veredicto

*(completar/ajustar después de revisar visualmente las candidatas de la sección 4.1; borrador basado en lo ya conocido del artículo V2 más lo encontrado en este EDA)*

- Dataset ya usado como base en IMJETA V2. Desbalance fuerte: Pneumonia (train 3,875) frente a COVID-19 (train 460) — razón ~8.4:1.
- Se encontraron duplicados exactos dentro del propio dataset, incluyendo algunos que cruzan train/val/test (fuga real) — confirma que la limpieza por hash MD5 que ya hace el pipeline de fusión (`01_dataset_preprocessing.ipynb`) es necesaria, no opcional.
- Se detectaron candidatas a baja calidad visual (fotos de la placa física con reflejo, recortes incompletos con bordes grandes en blanco/negro) — pendiente de revisión manual y decisión documentada sobre si se excluyen, y si se conserva una submuestra como "caso de robustez".
- Sin metadata demográfica propia; la heterogeneidad de población por clase (pediátrico vs adulto) es conocida solo por los papers de origen, no por los archivos.
- Aporta: la totalidad de Normal/Pneumonia del proyecto (no hay otra fuente de Pneumonia todavía) y una fracción de Tuberculosis/COVID-19 que se complementa con TBX11K-simplified/COVID-QU-Ex.
- Pendiente de decidir (ver `plan_biomedinformatics.md`): si se diversifica Pneumonia con otra fuente adulta, y si se recorta el total de Pneumonia para no dominar el pool combinado.